# PrimeNet — NF in-domain only (Colab, chunked)

Pretrain on **NF**, then finetune on **NF** (no mimic_all).

| Chunk | What |
|-------|------|
| **A** | SSL pretrain `fig5_pt_cohort` |
| **B** | Finetune `fig5_cohort_all` (all layers) |
| **C** | Finetune `fig5_cohort_final` (frozen) |

After each chunk: download `nf_progress.zip`. Next session: restore it.

**Upload to `/content/uploads/`:** `nf_saved_data.zip` from Desktop.

## 1. Clone + install

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "AhmedSofan10/ChemoTreeVsDL"
BRANCH = "primenet"
REPO_DIR = "/content/ChemoTreeVsDL"

if not Path(REPO_DIR).is_dir():
    subprocess.check_call(["git", "clone", "-b", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])

os.chdir(REPO_DIR)
sys.path.insert(0, str(Path.cwd()))
os.environ["PYTHONPATH"] = str(Path.cwd())
os.environ["PYTHONUNBUFFERED"] = "1"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pandas>=2.3.0"])

import torch
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Options

In [ ]:
from pathlib import Path

RUN_FAST = False
COHORT = "mimic_cohort_NF_30_days"
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

NF_CKPT = Path(
    f"MIMIC_IV/saved_data/results/{COHORT}/time_series/pretrain/primenet/"
    "fig5_pt_cohort/fold_0/grid_none"
)
print("Upload nf_saved_data.zip (+ optional nf_progress.zip) to", UPLOAD_DIR)

## 3. Place NF data (+ restore progress if any)

In [ ]:
import shutil
import zipfile
from pathlib import Path
from google.colab import files

print("Upload nf_saved_data.zip and optionally nf_progress.zip")
uploaded = files.upload()
for name in uploaded:
    src = Path(name)
    dst = UPLOAD_DIR / src.name
    if src.resolve() != dst.resolve():
        shutil.move(str(src), dst)
    print("staged", dst)

root = Path.cwd()
data_zips = list(UPLOAD_DIR.glob("*nf_saved_data*.zip")) + list(UPLOAD_DIR.glob("nf_saved_data.zip"))
if not data_zips:
    data_zips = [p for p in UPLOAD_DIR.glob("*.zip") if "progress" not in p.name.lower()]
if not data_zips:
    raise FileNotFoundError("Upload nf_saved_data.zip to /content/uploads/")

print("Extracting", data_zips[0])
with zipfile.ZipFile(data_zips[0]) as zf:
    zf.extractall(root)

prog = list(UPLOAD_DIR.glob("*nf_progress*.zip"))
if prog:
    print("Restoring progress", prog[0])
    with zipfile.ZipFile(prog[0]) as zf:
        zf.extractall(root / "MIMIC_IV" / "saved_data" / "results")

checks = [
    Path(f"MIMIC_IV/saved_data/cohorts/{COHORT}.csv.gz"),
    Path(f"MIMIC_IV/saved_data/top_features/mimic_top100_features.pkl"),
    Path(
        f"MIMIC_IV/saved_data/processed_admission_features_for_ts/{COHORT}/"
        f"{COHORT}_admissions_labs_14_days_to_ts.csv.gz"
    ),
    *[Path(f"MIMIC_IV/saved_data/folds/{COHORT}/fold_{i}.pkl") for i in range(5)],
]
missing = [str(p) for p in checks if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing:\n  - " + "\n  - ".join(missing))
print("OK — NF data ready")

## 4A. Chunk A — SSL pretrain on NF

In [ ]:
import subprocess
import sys
from pathlib import Path

def ckpt_ready(p: Path) -> bool:
    return (p / "checkpoint_best.bin").is_file() and (p / "primenet_saved_variables.pkl").is_file()

if ckpt_ready(NF_CKPT):
    print("SKIP — NF pretrain exists:", NF_CKPT)
    for name in ("checkpoint_best.bin", "primenet_saved_variables.pkl"):
        p = NF_CKPT / name
        print(f"  {name}: {p.stat().st_size/1e6:.2f} MB")
else:
    cmd = [
        sys.executable, "-u", "-m", "ts_model_training.main",
        "--dataset", "MIMIC_IV", "--cohort", COHORT, "--fold", "0",
        "--model_type", "primenet", "--grid", "none",
        "--feature_threshold", "--static_threshold", "0", "--hid_dim_demo", "64",
        "--config_path", "config/ts_config_params.yaml",
        "--prefix", "fig5_pt_cohort", "--pretrain",
    ]
    if RUN_FAST:
        cmd.append("--fast")
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)
print("CHUNK A done")

## 4B / 4C. Finetune chunks (edit PREFIX + FREEZE)

| Chunk | PREFIX | FREEZE |
|-------|--------|--------|
| **B** | `fig5_cohort_all` | `False` |
| **C** | `fig5_cohort_final` | `True` |

In [ ]:
import subprocess
import sys
from pathlib import Path

# --- EDIT THESE TWO LINES ---
PREFIX = "fig5_cohort_all"   # or fig5_cohort_final
FREEZE = False               # True for final
# ----------------------------

def fold_done(prefix, fold):
    p = Path(
        f"MIMIC_IV/saved_data/results/{COHORT}/time_series/finetune/primenet/"
        f"{prefix}/fold_{fold}/grid_none/results_final.csv"
    )
    return p.is_file()

if not ckpt_ready(NF_CKPT):
    raise FileNotFoundError(f"Run Chunk A first. Missing: {NF_CKPT}")

for fold in range(5):
    if fold_done(PREFIX, fold):
        print(f"SKIP fold {fold}")
        continue
    cmd = [
        sys.executable, "-u", "-m", "ts_model_training.main",
        "--dataset", "MIMIC_IV", "--cohort", COHORT, "--fold", str(fold),
        "--model_type", "primenet", "--grid", "none",
        "--feature_threshold", "--static_threshold", "0", "--hid_dim_demo", "64",
        "--config_path", "config/ts_config_params.yaml",
        "--prefix", PREFIX, "--load_ckpt_path", str(NF_CKPT),
    ]
    if FREEZE:
        cmd.append("--freeze")
    if RUN_FAST:
        cmd.append("--fast")
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

print("CHUNK done:", PREFIX)

## 5. Save progress (run after every chunk)

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

src = Path("MIMIC_IV/saved_data/results")
archive = shutil.make_archive("/content/nf_progress", "zip", root_dir=src)
print("Created", archive, f"({Path(archive).stat().st_size/1e6:.1f} MB)")
files.download(archive)

## 6. Summarize (after B + C done)

In [ ]:
import csv
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(f"MIMIC_IV/saved_data/results/{COHORT}/time_series/finetune/primenet")
SCENARIOS = {
    "cohort/all": "fig5_cohort_all",
    "cohort/final": "fig5_cohort_final",
}
METRICS = ["auroc", "auprc", "f1", "precision", "recall", "loss"]

def read_test(p: Path) -> dict:
    with open(p) as f:
        rows = list(csv.DictReader(f))
    test = [r for r in rows if str(r.get("split", "")).lower() == "test"]
    return test[-1]

rows = []
for label, prefix in SCENARIOS.items():
    for fold in range(5):
        path = ROOT / prefix / f"fold_{fold}" / "grid_none" / "results_final.csv"
        if not path.is_file():
            print("MISSING", path)
            continue
        r = read_test(path)
        rows.append({"scenario": label, "fold": fold, **{m: float(r[m]) for m in METRICS if m in r and r[m] != ""}})

df = pd.DataFrame(rows)
print("=== Per-fold ===")
display(df.round(4) if not df.empty else df)
print("\n=== Mean ± std ===")
if not df.empty:
    display(df.groupby("scenario")[METRICS].agg(["mean", "std", "count"]).round(4))
print("\n=== Paper-style (NF in-domain) ===")
for scen, g in df.groupby("scenario"):
    parts = [f"{m.upper()} {g[m].mean():.3f}±{g[m].std(ddof=1):.3f}" for m in ["auroc", "auprc", "f1"] if m in g]
    print(f"{scen:16s}  " + "  ".join(parts) + f"  (n={len(g)})")